In [18]:

import jax
import jax.numpy as jnp
import flax.nnx as nnx
import netket as nk
import netket.experimental as nkx
import sys
sys.path.append('..')
from NES_VMC import NESTotalAnsatz, create_machine,init_sampler_state,\
    generate_random_initial_states,ha,SingleStateAnsatz,create_single_machine,\
        create_machine_matrix,Ham_psi,Ham_Psi,NES_loss_energy,nes_vmc_gradient,hi,E_fcis,mcmc_sampler_multichain,\
            compute_qgt
import optax
from typing import Callable
from functools import partial
from jax.flatten_util import ravel_pytree
import time
from collections import Counter
import numpy as np
K=2
hi_ext = hi**K

In [22]:
N_CHAINS = 16
N_WARMUP = 100
N_SAMPLES_PER_CHAIN = 200
SWEEP_SIZE = 30
N_ITER =50

rngs = nnx.Rngs(42)
total_ansatz = NESTotalAnsatz(4, n_states=K, hidden_dim=12, rngs=rngs)
single_ansatz = SingleStateAnsatz(4, hidden_dim=8, rngs=rngs)
total_machine, total_graphdef, total_params = create_machine(total_ansatz)
total_matrix_machine, total_graphdef, total_params = create_machine_matrix(total_ansatz)

single_machine_list = []
for ansatz in total_ansatz.single_ansatz_list:
    m, g, p = create_single_machine(ansatz)
    single_machine_list.append(m)
    
optimizer = optax.sgd(learning_rate=0.01)
opt_state = optimizer.init(total_params)

# ===================== 7. 训练循环（多链版本） =====================
print("\n" + "="*60)
print("开始多链 NES-VMC 训练 (朴素梯度下降法)")
print("="*60)

history = {
    'step': [],
    'energy': [],
    'energy_std': [],
    'loss': [],
    'params': [],
    'E_Lmatrix':[],
    'natural_grad':[],
    'grad_flat':[],
    'samples':[],
    'log_Psi':[],
    'log_M':[]
}
print(f"基态能量={E_fcis[0]:.8f} Ha| 第一激发态能量={E_fcis[1]:.8f} Ha| 第二激发态能量={E_fcis[2]:.8f} Ha")
sampler_state = init_sampler_state(hi_ext, N_CHAINS, seed=21)  # 每次迭代换种子避免初始状态固定
start_time = time.time()
for step in range(N_ITER):
    # 1. 生成多链随机初始状态（模仿NetKet，无需手动指定单个initial_state）
    # 2. 多链采样（总样本数=16*63=1008，和原单链一致）
    samples,sampler_state = mcmc_sampler_multichain(
        n_samples_per_chain=N_SAMPLES_PER_CHAIN,
        n_warmup=N_WARMUP,
        sampler_state=sampler_state,
        edges=((0,1),(2,3),(4,5),(6,7)),
        machine=total_machine,
        params=total_params,
    )
    #samples = samples.reshape(-1,2,4)

    # 3. 计算能量和自然梯度（逻辑和原代码一致）
    grad, loss_mean, E_L_mean = nes_vmc_gradient(ha=ha,
                                                 total_matrix_machine=total_matrix_machine,
                                                 total_machine=total_machine,
                                                 single_machine_list=single_machine_list,
                                                 total_params=total_params,
                                                 x_batch=samples.reshape(-1,K,4))
    #grad = jax.tree_util.tree_map(lambda x: x * 2, grad)
    
    grad_flat , grad_unravel_fn = ravel_pytree(grad)
    # qgt_reg, unravel_fn = compute_qgt(total_machine, total_params, samples.reshape(-1,2,4), diag_shift=0.1)
    
    # # # 自然梯度求解
    # natural_grad_flat = jnp.linalg.solve(qgt_reg, grad_flat)
    # natural_grad = grad_unravel_fn(natural_grad_flat)
    # grad = natural_grad
        
    # 4. 更新参数
    updates, opt_state = optimizer.update(grad, opt_state, total_params)
    total_params = optax.apply_updates(total_params, updates)
    
    # 5. 记录历史
    if step % 5 == 0 or step == N_ITER - 1:
        # --------------------- 【NES-VMC 监控模板】直接用 ---------------------
        # 1. 监控 log_Psi
        log_Psi_batch = total_machine(total_params, samples.reshape(-1,K,4))
        print(f"log_Psi: mean={log_Psi_batch.mean():.3f} | min={log_Psi_batch.min():.3f} | max={log_Psi_batch.max():.3f}")

        # 2. 监控梯度范数
        grad_norm = jnp.linalg.norm(grad_flat)
        print(f"grad norm = {grad_norm:.4f}")

        # 5. 局域能量矩阵
        print(f"E_L mean =\n{E_L_mean}")
    
        eig_vals, eig_vecs = jnp.linalg.eigh(E_L_mean)
        history['step'].append(step)
        history['E_Lmatrix'].append(E_L_mean)
        history['samples'].append(samples)
        history['loss'].append(loss_mean)
        # #history['natural_grad'].append(natural_grad)
        # history['grad_flat'].append(grad_flat)
        # history['log_Psi'].append(log_Psi)
        # history['log_M'].append(log_M)
        history['params'].append(total_params)
        print(f"Step {step:3d} | Loss: {loss_mean}|0st能量={eig_vals[0]:.8f} Ha| 1st能量={eig_vals[1]:.8f} Ha")
        # print(f'grad={grad_flat[30:31]}')
        print('#-----------------------------------------#')


end_time = time.time()
print(f"训练耗时：{end_time - start_time:.2f} 秒")
# 最终结果
print("\n" + "="*60)
print(f"训练完成!")
# print(f"最终能量：{final_energy.real:.8f} ± {final_std:.6f} Ha")
# print(f"FCI 基准：{E_fcis[0]:.8f} Ha")
# print(f"绝对误差：{final_error:.6f} Ha")
# print(f"相对误差：{final_error / jnp.abs(E_fcis[0]) * 100:.4f}%")
print("="*60)


开始多链 NES-VMC 训练 (朴素梯度下降法)
基态能量=-1.01546825 Ha| 第一激发态能量=-0.87542794 Ha| 第二激发态能量=-0.42938376 Ha
log_Psi: mean=0.889-0.570j | min=-0.622-0.434j | max=1.233+1.152j
grad norm = 0.8639
E_L mean =
[[-0.77968689-1.34803085e-01j  0.23849937-2.06282208e-01j]
 [ 0.25240582+8.11126141e-05j -0.39219073+1.34558826e-01j]]
Step   0 | Loss: -1.1718776127500679|0st能量=-0.91522857 Ha| 1st能量=-0.25664904 Ha
#-----------------------------------------#
log_Psi: mean=1.053-0.419j | min=-0.757-0.471j | max=1.411+1.179j
grad norm = 0.8876
E_L mean =
[[-0.77452489-0.09844509j  0.31071342-0.10569706j]
 [ 0.27653309-0.00370868j -0.4735187 +0.09977624j]]
Step   5 | Loss: -1.2480435916134913|0st能量=-0.95788734 Ha| 1st能量=-0.29015625 Ha
#-----------------------------------------#
log_Psi: mean=0.840-0.257j | min=-0.799-0.497j | max=1.241+1.266j
grad norm = 6.1468
E_L mean =
[[-0.7176293 +0.27369418j  0.13405002+0.6538047j ]
 [ 0.13142188-0.1931566j  -0.49650645-0.27247704j]]
Step  10 | Loss: -1.2141357517719262|0st能量=-

In [17]:
N_CHAINS = 16
N_WARMUP = 100
N_SAMPLES_PER_CHAIN = 200
SWEEP_SIZE = 30
N_ITER =50

rngs = nnx.Rngs(42)
total_ansatz = NESTotalAnsatz(4, n_states=K, hidden_dim=8, rngs=rngs)
single_ansatz = SingleStateAnsatz(4, hidden_dim=8, rngs=rngs)
total_machine, total_graphdef, total_params = create_machine(total_ansatz)
total_matrix_machine, total_graphdef, total_params = create_machine_matrix(total_ansatz)

single_machine_list = []
for ansatz in total_ansatz.single_ansatz_list:
    m, g, p = create_single_machine(ansatz)
    single_machine_list.append(m)
    
optimizer = optax.sgd(learning_rate=0.01)
opt_state = optimizer.init(total_params)

# ===================== 7. 训练循环（多链版本） =====================
print("\n" + "="*60)
print("开始多链 NES-VMC 训练 (自然梯度下降法)")
print("="*60)

history = {
    'step': [],
    'energy': [],
    'energy_std': [],
    'loss': [],
    'params': [],
    'E_Lmatrix':[],
    'natural_grad':[],
    'grad_flat':[],
    'samples':[],
    'log_Psi':[],
    'log_M':[]
}
print(f"基态能量={E_fcis[0]:.8f} Ha| 第一激发态能量={E_fcis[1]:.8f} Ha| 第二激发态能量={E_fcis[2]:.8f} Ha")
sampler_state = init_sampler_state(hi_ext, N_CHAINS, seed=21)  # 每次迭代换种子避免初始状态固定
start_time = time.time()
for step in range(N_ITER):
    # 1. 生成多链随机初始状态（模仿NetKet，无需手动指定单个initial_state）
    # 2. 多链采样（总样本数=16*63=1008，和原单链一致）
    samples,sampler_state = mcmc_sampler_multichain(
        n_samples_per_chain=N_SAMPLES_PER_CHAIN,
        n_warmup=N_WARMUP,
        sampler_state=sampler_state,
        edges=((0,1),(2,3),(4,5),(6,7)),
        machine=total_machine,
        params=total_params,
    )
    #samples = samples.reshape(-1,2,4)

    # 3. 计算能量和自然梯度（逻辑和原代码一致）
    grad, loss_mean, E_L_mean = nes_vmc_gradient(ha=ha,
                                                 total_matrix_machine=total_matrix_machine,
                                                 total_machine=total_machine,
                                                 single_machine_list=single_machine_list,
                                                 total_params=total_params,
                                                 x_batch=samples.reshape(-1,K,4))
    #grad = jax.tree_util.tree_map(lambda x: x * 2, grad)
    
    grad_flat , grad_unravel_fn = ravel_pytree(grad)
    qgt_reg, unravel_fn = compute_qgt(total_machine, total_params, samples.reshape(-1,2,4), diag_shift=0.1)
    
    # # 自然梯度求解
    natural_grad_flat = jnp.linalg.solve(qgt_reg, grad_flat)
    natural_grad = grad_unravel_fn(natural_grad_flat)
    grad = natural_grad
        
    # 4. 更新参数
    updates, opt_state = optimizer.update(grad, opt_state, total_params)
    total_params = optax.apply_updates(total_params, updates)
    
    # 5. 记录历史
    if step % 5 == 0 or step == N_ITER - 1:
        # total_model =  nnx.merge(graphdef,total_params)
        # log_Psi,log_M  = total_model(samples.reshape(-1,2,4))
        # --------------------- 【NES-VMC 监控模板】直接用 ---------------------
        # 1. 监控 log_Psi
        log_Psi_batch = total_machine(total_params, samples.reshape(-1,K,4))
        print(f"log_Psi: mean={log_Psi_batch.mean():.3f} | min={log_Psi_batch.min():.3f} | max={log_Psi_batch.max():.3f}")

        # 2. 监控梯度范数
        grad_norm = jnp.linalg.norm(grad_flat)
        print(f"grad norm = {grad_norm:.4f}")

        # 3. 监控自然梯度范数
        nat_grad_norm = jnp.linalg.norm(natural_grad_flat)
        print(f"nat grad norm = {nat_grad_norm:.4f}")

        # 4. 监控 QGT 条件数（判断是否奇异）
        cond = jnp.linalg.cond(qgt_reg)
        print(f"QGT cond = {cond:.2e}")

        # 5. 局域能量矩阵
        print(f"E_L mean =\n{E_L_mean}")
    
        eig_vals, eig_vecs = jnp.linalg.eigh(E_L_mean)
        history['step'].append(step)
        history['E_Lmatrix'].append(E_L_mean)
        history['samples'].append(samples)
        history['loss'].append(loss_mean)
        # #history['natural_grad'].append(natural_grad)
        # history['grad_flat'].append(grad_flat)
        # history['log_Psi'].append(log_Psi)
        # history['log_M'].append(log_M)
        history['params'].append(total_params)
        print(f"Step {step:3d} | Loss: {loss_mean}|基态能量={eig_vals[0]:.8f} Ha| 第一激发态能量={eig_vals[1]:.8f} Ha Ha")
        print(f'grad={grad_flat[30:31]}')
        print('#################################')


end_time = time.time()
print(f"训练耗时：{end_time - start_time:.2f} 秒")
# 最终结果
print("\n" + "="*60)
print(f"训练完成!")
# print(f"最终能量：{final_energy.real:.8f} ± {final_std:.6f} Ha")
# print(f"FCI 基准：{E_fcis[0]:.8f} Ha")
# print(f"绝对误差：{final_error:.6f} Ha")
# print(f"相对误差：{final_error / jnp.abs(E_fcis[0]) * 100:.4f}%")
print("="*60)


开始多链 NES-VMC 训练 (自然梯度下降法)
基态能量=-1.01546825 Ha| 第一激发态能量=-0.87542794 Ha| 第二激发态能量=-0.42938376 Ha
log_Psi: mean=0.813-0.033j | min=-0.390-2.799j | max=0.954+1.865j
grad norm = 0.5843
nat grad norm = 2.2009
QGT cond = 5.24e+01
E_L mean =
[[-0.68678526-0.09697106j  0.24823534-0.62342476j]
 [ 0.05179148+0.10482586j -0.62499684+0.09668739j]]
Step   0 | Loss: -1.3117820926748878|基态能量=-1.05091727 Ha| 第一激发态能量=-0.26086482 Ha Ha
grad=[0.00221615+0.04483759j]
#################################
log_Psi: mean=0.838-0.050j | min=-0.576-2.056j | max=1.129+1.756j
grad norm = 0.3165
nat grad norm = 1.3518
QGT cond = 4.91e+01
E_L mean =
[[-0.71758786-0.0723694j   0.2234645 -0.52285543j]
 [ 0.06170392+0.11849763j -0.62803813+0.07092j   ]]
Step   5 | Loss: -1.3456259866072133|基态能量=-1.02660457 Ha| 第一激发态能量=-0.31902141 Ha Ha
grad=[-0.00467022+0.00450997j]
#################################
log_Psi: mean=0.922-0.118j | min=-0.375-1.223j | max=1.244+1.635j
grad norm = 0.3633
nat grad norm = 1.3883
QGT cond = 5.03e

In [11]:
def sampler_info(samples:jnp.array,K:int):
    test_samples = np.array(samples.reshape(-1, 4*K))
    count = Counter(tuple(each_row.tolist()) for each_row in test_samples)
    for tpl, count_ in count.items():
        print(f"元组 {tpl} 出现了 {count_} 次")
    return count

In [14]:
s = sampler_info(history['samples'][5],K)


元组 (1, 0, 1, 0, 1, 0, 0, 1) 出现了 1336 次
元组 (1, 0, 0, 1, 1, 0, 1, 0) 出现了 1469 次
元组 (0, 1, 1, 0, 1, 0, 0, 1) 出现了 93 次
元组 (1, 0, 1, 0, 0, 1, 0, 1) 出现了 66 次
元组 (1, 0, 0, 1, 0, 1, 1, 0) 出现了 121 次
元组 (0, 1, 0, 1, 1, 0, 1, 0) 出现了 63 次
元组 (1, 0, 1, 0, 0, 1, 1, 0) 出现了 16 次
元组 (0, 1, 0, 1, 1, 0, 0, 1) 出现了 10 次
元组 (0, 1, 1, 0, 0, 1, 0, 1) 出现了 7 次
元组 (1, 0, 0, 1, 0, 1, 0, 1) 出现了 4 次
元组 (0, 1, 1, 0, 1, 0, 1, 0) 出现了 13 次
元组 (0, 1, 0, 1, 0, 1, 1, 0) 出现了 2 次


In [15]:
log_Psi_batch = total_machine(total_params, history['samples'][0].reshape(-1,K,4))
print(f"log_Psi: mean={log_Psi_batch.mean():.3f} | min={log_Psi_batch.min():.3f} | max={log_Psi_batch.max():.3f}")

log_Psi: mean=1.162+1.049j | min=-0.419-0.167j | max=2.302+3.123j
